# 01 — Attribution Graph Generation

This notebook generates attribution graphs from language models using Anthropic's
open-source circuit-tracer library. We generate graphs for diverse prompt types
and also create synthetic graphs for development/validation.

**Steps:**
1. Generate synthetic attribution graphs (for development)
2. Load prompt sets across task categories
3. Generate real attribution graphs from Gemma-2-2B
4. Visualize sample graphs from each category

In [1]:
import sys
sys.path.insert(0, "..")

import json
import os
import importlib
from pathlib import Path

# Force reload
import src.graph_generator
import src.utils
importlib.reload(src.graph_generator)
importlib.reload(src.utils)

from src.graph_generator import (
    SyntheticGraphGenerator,
    CircuitTracerGenerator,
    AttributionGraph,
)
from src.utils import visualize_attribution_graph

print("Modules loaded (fresh reload).")


Modules loaded (fresh reload).


## 1. Generate Synthetic Data

We first create synthetic graphs with known structural properties.
This lets us validate our metrics pipeline and train initial models
before investing time in expensive model inference.

Three types:
- **Clean trees** (label=1.0): Simulate interpretable circuits with clear hierarchy
- **Tangled graphs** (label=0.0): Simulate superposition with dense cross-connections
- **Mixed graphs** (label=0.5): Partial superposition (some clean regions, some tangled)

In [2]:
synth_gen = SyntheticGraphGenerator()

# Generate the synthetic dataset
synthetic_graphs = synth_gen.generate_dataset(
    n_clean=150,
    n_tangled=150,
    n_mixed=75,
    output_dir='../data/raw/synthetic',
)

print(f'\nTotal synthetic graphs: {len(synthetic_graphs)}')
print(f'  Clean (interpretable): 150')
print(f'  Tangled (superposition): 150')
print(f'  Mixed (partial): 75')

Generated 375 synthetic graphs -> ../data/raw/synthetic

Total synthetic graphs: 375
  Clean (interpretable): 150
  Tangled (superposition): 150
  Mixed (partial): 75


In [3]:
# Visualize one of each type
clean_example = synth_gen.generate_clean_tree(depth=4, branching_factor=3)
tangled_example = synth_gen.generate_tangled_graph(n_nodes=30, edge_density=0.12)
mixed_example = synth_gen.generate_mixed_graph()

visualize_attribution_graph(
    clean_example,
    title='Clean Tree (Simulated Interpretable Circuit)',
    save_path='../results/figures/synthetic_clean.png',
)

visualize_attribution_graph(
    tangled_example,
    title='Tangled Graph (Simulated Superposition)',
    save_path='../results/figures/synthetic_tangled.png',
)

visualize_attribution_graph(
    mixed_example,
    title='Mixed Graph (Partial Superposition)',
    save_path='../results/figures/synthetic_mixed.png',
)

Saved figure to ../results/figures/synthetic_clean.png
Saved figure to ../results/figures/synthetic_tangled.png
Saved figure to ../results/figures/synthetic_mixed.png


## 2. Load Prompt Sets

We use curated prompts across 6 categories. Our hypothesis is that
different task types will produce structurally different attribution graphs.

In [4]:
with open('../data/prompts/prompt_sets.json', 'r') as f:
    prompt_data = json.load(f)

for category, data in prompt_data.items():
    if category == 'description':
        continue
    print(f'{category}: {len(data["prompts"])} prompts — {data["description"]}')

factual_recall: 20 prompts — Simple factual questions — expect clean, interpretable circuits
reasoning: 20 prompts — Multi-step reasoning — expect more complex, deeper circuits
creative_writing: 20 prompts — Creative generation — expect planning circuits (rhyme, narrative)
code_understanding: 15 prompts — Code-related prompts — tests structured/algorithmic reasoning
ambiguous_context: 20 prompts — Ambiguous or polysemous prompts — likely to trigger superposition
multilingual: 10 prompts — Mixed-language prompts — tests cross-lingual feature activation


## 3. Generate Real Attribution Graphs

Using Anthropic's circuit-tracer with Gemma-2-2B.


In [5]:
# Initialize generator

generator = CircuitTracerGenerator(
    model_name='google/gemma-2-2b',
    device='auto',
    backend='transformerlens',
)

generator.load_model()

Loading google/gemma-2-2b with transcoder_set='gemma' on mps (transformerlens backend)...


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded successfully.


In [6]:
# Generate graphs for each prompt category
# using smaller max_nodes (200) and batch_size (64) for speed

all_real_graphs = []

for category, data in prompt_data.items():
    if category == 'description':
        continue
    
    print(f'\n{"="*60}')
    print(f'Category: {category}')
    print(f'{"="*60}')

    prompts = data['prompts'][:5]
    
    output_dir = f'../data/raw/{category}'
    graphs = generator.generate_batch(
        prompts=prompts,
        output_dir=output_dir,
        node_threshold=0.8,
        edge_threshold=0.98,
        max_nodes=200,
        batch_size=64,
    )
    all_real_graphs.extend(graphs)

print(f'\nTotal real attribution graphs generated: {len(all_real_graphs)}')

Phase 0: Precomputing activations and vectors



Category: factual_recall
[1/5] Generating graph for: The capital of France is...


Precomputation completed in 10.66s
Found 4420 active features
Phase 1: Running forward pass
Forward pass completed in 581.84s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6953
Will include 200 of 4420 feature nodes
Input vectors built in 0.90s
Phase 3: Computing logit attributions
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/circuit_tracer/attribution/context_transformerlens.py:223: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 12.87s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 200/200 [00:49<00:00,  4.04it/s]
Feature attributions completed in 49.45s
Attribution completed


DEBUG: Introspecting circuit-tracer Graph object
  activation_values: Tensor shape=torch.Size([4420]) dtype=torch.bfloat16
  active_features: Tensor shape=torch.Size([4420, 3]) dtype=torch.int64
  adjacency_matrix: Tensor shape=torch.Size([372, 372]) dtype=torch.float32
  cfg: UnifiedConfig
  input_string: <bos>The capital of France is
  input_tokens: Tensor shape=torch.Size([6]) dtype=torch.int64
  logit_probabilities: Tensor shape=torch.Size([10]) dtype=torch.bfloat16
  logit_targets: list len=10
    [0] = LogitTarget(token_str=' a', vocab_idx=476)
  logit_token_ids: Tensor shape=torch.Size([10]) dtype=torch.int64
  logit_tokens: Tensor shape=torch.Size([10]) dtype=torch.int64
  n_pos: 6
  scan: mwhanna/gemma-scope-transcoders
  selected_features: Tensor shape=torch.Size([200]) dtype=torch.int64
  vocab_size: 256000

  adjacency_matrix shape: torch.Size([372, 372])
  node_mask shape: torch.Size([372]), sum=150
  edge_mask shape: torch.Size([372, 372]), sum=5710
  active_features sha

Precomputation completed in 21.76s
Found 8636 active features
Phase 1: Running forward pass
Forward pass completed in 968.55s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6602
Will include 200 of 8636 feature nodes
Input vectors built in 1.12s
Phase 3: Computing logit attributions
Logit attributions completed in 19.54s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 200/200 [01:15<00:00,  2.63it/s]
Feature attributions completed in 75.97s
Attribution completed in 1086.94s
Phase 0: Precomputing activations and vectors


...

  -> 291 nodes, 27418 edges (2689.0s)
[5/5] Generating graph for: If you rearrange the letters CIFAIPC you get the word...


Precomputation completed in 31.34s
Found 13453 active features
Phase 1: Running forward pass
Forward pass completed in 1384.03s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.4102
Will include 200 of 13453 feature nodes
Input vectors built in 1.15s
Phase 3: Computing logit attributions
Logit attributions completed in 28.40s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 200/200 [01:53<00:00,  1.77it/s]
Feature attributions completed in 113.08s
Attribution completed in 1558.02s
Phase 0: Precomputing activations and vectors


  Adjacency matrix: 561 nodes
  Layout candidates: {'selected+error+embed+logit': 561, 'selected+embed+logit': 223, 'selected+logit': 210, 'selected_only': 200, 'active+error+embed+logit': 13814}
  Matched layout: selected+error+embed+logit
  -> 219 nodes, 16635 edges (1558.1s)

Generated 5/5 graphs successfully.

Category: creative_writing
[1/5] Generating graph for: Once upon a time in a land far away, there lived a...


Precomputation completed in 34.89s
Found 15327 active features
Phase 1: Running forward pass


KeyboardInterrupt: 

In [ ]:
# Visualize a sample from each category
for category, data in prompt_data.items():
    if category == 'description':
        continue
    
    graph_dir = Path(f'../data/raw/{category}')
    graph_files = sorted(graph_dir.glob('graph_*.json'))
    
    if graph_files:
        sample = AttributionGraph.load(str(graph_files[0]))
        visualize_attribution_graph(
            sample,
            title=f'{category}: "{sample.prompt[:50]}..."',
            save_path=f'../results/figures/sample_{category}.png',
        )